# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*  

The queue below ranks pages by `predicted_high_gap` (Model E's output, using a train-only
`expected_ctr` per tier, per w06). Each row carries a reason code explaining *why* it's flagged
and a plain-language action.

`page_3_5` pages are never scored as high-gap under any circumstance — a limitation found in
w06 (the tier's `expected_ctr` sits below the flagging threshold, so `ctr_gap` can never cross
it). These pages get `NOT_EVALUABLE_LABEL_LIMITATION` and route to manual review instead of an
automated recommendation.

For evaluable pages (`page_1`, `striking`) with `is_high_gap == 1`, reason codes split on two
medians computed from the flagged pool: `ctr_gap` (severity of the gap) and `search_volume`
(traffic at stake):

- `HIGH_PRIORITY_CTR_GAP` — severe gap, high volume
- `SEVERE_CTR_GAP_LOW_VOLUME` — severe gap, low volume
- `HIGH_VOLUME_LOW_CTR_GAP` — mild gap, high volume
- `LOW_PRIORITY_CTR_GAP` — mild gap, low volume

Pages with `is_high_gap == 0` (outside `page_3_5`) need no action and are left out of the queue.

Given w06's finding that a single split's precision@20 ranged 40-95% (mean ~58%) across seeds,
this ranking should be read as directional prioritization, not a confirmed-accurate ordering.  

*Cost/value note:* *`search_volume` stands in for potential value (traffic at stake), and `competition_level` stands in for likely effort — so `HIGH_PRIORITY_CTR_GAP` pages with `LOW competition` are the cheapest wins, while `HIGH competition` pages may need more effort for the same urgency.*

In [ ]:
import os
from huggingface_hub import login
import duckdb
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

fact_content_daily_performance_table = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")
dim_content_table = con.sql(f"SELECT * from read_parquet('{rel}/dim_content.parquet')")

df = con.sql("""SELECT f.client_hash_id, f.content_hash_id, SUM(f.gsc_clicks) AS total_clicks, SUM(f.gsc_impressions) AS total_impressions,
                SUM(f.gsc_avg_position * f.gsc_impressions) *1.0 / SUM(f.gsc_impressions) AS weighted_avg_position,
                ANY_VALUE(d.search_volume) AS search_volume, ANY_VALUE(d.competition_level) AS competition_level, ANY_VALUE(d.main_intent) AS main_intent
                FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING(content_hash_id)
                WHERE
                    f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND f.gsc_avg_position >0 AND d.search_volume IS NOT NULL AND d.competition_level IS NOT NULL
                GROUP BY f.client_hash_id, f.content_hash_id ORDER BY f.client_hash_id, f.content_hash_id""").df()

df = df.query("total_impressions >= 200").reset_index(drop=True)

df["main_intent"] = df["main_intent"].astype("category")
df["ctr"] = df["total_clicks"] / df["total_impressions"]
competition_ranks = pd.api.types.CategoricalDtype(categories=["LOW", "MEDIUM", "HIGH"], ordered=True)
df["competition_level"] = df["competition_level"].astype(competition_ranks)
df["position_tier"] = pd.cut(df["weighted_avg_position"], bins=[0,10,20,float("inf")], labels=["page_1", "striking", "page_3_5"])

competition_map = {"LOW": 0, "MEDIUM": 1, "HIGH":2}
df["competition_level_enc"] = df["competition_level"].map(competition_map)

ohe = OneHotEncoder(sparse_output=False)
categories = ["position_tier"]
#fit it on the training data to learn the categories and transforms it (meaning it starts filling up the matrix with 1s and 0s)
position_encoded = ohe.fit_transform(df[categories])

#get the column names, reason why it was on ohe not position_encoded because position_encoded is a matrix without column names, ohe creates the column names
position_columns = ohe.get_feature_names_out(categories)

#the reason for index= df.index is to ensure the dataframe doesnt mix up cuz it will be added to df, so it must have it's index
position_df = pd.DataFrame(data=position_encoded, columns=position_columns, index=df.index)
df[position_columns] = position_df

#basically because search_volume is heavily skewed, we use log for it so that the model doesnt depend too much on it basically
#log1p(x) means log(1+x), reason for the 1+ is cuz x can be 0, to not get a math error
df["log_search_volume"] = np.log1p(df["search_volume"])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display

feature_list = ["weighted_avg_position", "log_search_volume", "competition_level_enc"] + position_columns.tolist()
X = df[feature_list]

groups = df["client_hash_id"]
seeds_data = {"seed": [], "train_split": [], "test_split": []}
best_seed = None
best_diff = float("inf")
best_fraction = None
for seed in range(1, 51):
    gss_try = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx_try, test_idx_try = next(gss_try.split(X, groups=groups))
    fraction_try = len(test_idx_try) / len(X)
    diff = abs(fraction_try - 0.2)
    if diff < 0.01:
        seeds_data["seed"].append(seed)
        seeds_data["train_split"].append(f"{len(train_idx_try) / len(X):%}")
        seeds_data["test_split"].append(f"{fraction_try:%}")
        
    if diff < best_diff:
        best_diff = diff
        best_seed = seed
        best_fraction = fraction_try

gss = GroupShuffleSplit(test_size =0.2, n_splits = 1, random_state = best_seed)
train_idx, test_idx = next(gss.split(X, groups= groups))

train_clients = set(df.iloc[train_idx]["client_hash_id"])
test_clients = set(df.iloc[test_idx]["client_hash_id"])

print(f"Clients shared by train and test(grouped model): {len(train_clients & test_clients)}")

X_E_train = X.iloc[train_idx]
X_E_test = X.iloc[test_idx]

expected_ctr_per_tier = df.loc[X_E_train.index].groupby("position_tier")["total_clicks"].sum() / df.loc[X_E_train.index].groupby("position_tier")["total_impressions"].sum()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_per_tier).astype(float)
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

truth_threshold = 0.002
df["is_high_gap"] = (df["ctr_gap"] >= truth_threshold).astype(int)

y = df["is_high_gap"]
y_E_train = y.iloc[train_idx]
y_E_test = y.iloc[test_idx]

model_E = LogisticRegression(max_iter=1000, random_state=1)
model_E.fit(X_E_train, y_E_train)
y_E_pred_proba = model_E.predict_proba(X_E_test)[:,1]

test_df_E = df.loc[X_E_test.index].copy()
test_df_E["is_high_gap"] = y_E_test.values
test_df_E["predicted_high_gap"] = y_E_pred_proba
test_df_model_E = test_df_E.sort_values("predicted_high_gap", ascending=False)

data = {"k": [], "Model E Precision@k": []}
for k in [20,50,100,200,500,1000]:
    data["k"].append(k)
    top_k = test_df_model_E.head(k)
    precision_at_k = top_k["is_high_gap"].sum()/k
    data["Model E Precision@k"].append(precision_at_k)

display(pd.DataFrame(data).style.hide(axis="index").format("{:.1%}", subset=["Model E Precision@k"]).set_properties(**{"text-align": "center"}))

Clients shared by train and test(grouped model): 0


k,Model E Precision@k
20,95.0%
50,78.0%
100,71.0%
200,73.0%
500,68.2%
1000,65.4%


In [49]:
ctr_gap_median = test_df_E.query("is_high_gap == 1 and position_tier != 'page_3_5'")["ctr_gap"].median()
search_volume_median = test_df_E.query("is_high_gap == 1 and position_tier != 'page_3_5'")["search_volume"].median()

print(f"ctr_gap median for all test pages that have high gap and arent at tier 'page_3_5': {ctr_gap_median}")
print(f"search_volume median for all test pages that have high gap and arent at tier 'page_3_5': {search_volume_median}")

severe = (test_df_E["ctr_gap"] >= ctr_gap_median)
high_volume = (test_df_E["search_volume"] >= search_volume_median)
conditions = [
    test_df_E["position_tier"] == "page_3_5",
    (test_df_E["is_high_gap"] == 1) & severe & high_volume,
    (test_df_E["is_high_gap"] == 1) & severe & ~high_volume,
    (test_df_E["is_high_gap"] == 1) & ~severe & high_volume,
    (test_df_E["is_high_gap"] == 1) & ~severe & ~high_volume
]

labels = [
    "NOT_EVALUABLE_LABEL_LIMITATION",
    "HIGH_PRIORITY_CTR_GAP",
    "SEVERE_CTR_GAP_LOW_VOLUME",
    "HIGH_VOLUME_LOW_CTR_GAP",
    "LOW_PRIORITY_CTR_GAP"
]

test_df_E["reason_code"] = np.select(conditions, labels, default="NO_ACTION_NEEDED")
filtered_test_df_model_E = test_df_E.query("reason_code != 'NO_ACTION_NEEDED'").copy()

action_map = {
    "NOT_EVALUABLE_LABEL_LIMITATION": "Manual review only — model score not valid for this tier.",
    "HIGH_PRIORITY_CTR_GAP": "Review first — large gap, meaningful traffic.",
    "SEVERE_CTR_GAP_LOW_VOLUME": "Review when time allows — large gap, low traffic.",
    "HIGH_VOLUME_LOW_CTR_GAP": "Watch — high traffic, gap not severe yet.",
    "LOW_PRIORITY_CTR_GAP": "Low priority — no traffic nor is the gap big, keep it at last",
}
filtered_test_df_model_E["action"] = filtered_test_df_model_E["reason_code"].map(action_map)
display(filtered_test_df_model_E.head(10))

ctr_gap median for all test pages that have high gap and arent at tier 'page_3_5': 0.003289557533093802
search_volume median for all test pages that have high gap and arent at tier 'page_3_5': 10.0


,client_hash_id,content_hash_id,total_clicks,total_impressions,weighted_avg_position,search_volume,competition_level,main_intent,ctr,position_tier,...,position_tier_page_1,position_tier_page_3_5,position_tier_striking,log_search_volume,expected_ctr,ctr_gap,is_high_gap,predicted_high_gap,reason_code,action
1,client_0797ff3a1fc9a6a5,content_1207efddce873942,0.0,461.0,14.488069,0,LOW,informational,0.000000,striking,...,0.0,0.0,1.0,0.000000,0.003290,0.003290,1,0.437291,SEVERE_CTR_GAP_LOW_VOLUME,"Review when time allows — large gap, low traffic."
2,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,0.0,232.0,11.961207,0,LOW,informational,0.000000,striking,...,0.0,0.0,1.0,0.000000,0.003290,0.003290,1,0.408200,SEVERE_CTR_GAP_LOW_VOLUME,"Review when time allows — large gap, low traffic."
3,client_0797ff3a1fc9a6a5,content_27f8100281413b37,0.0,464.0,9.437500,20,HIGH,transactional,0.000000,page_1,...,1.0,0.0,0.0,3.044522,0.003711,0.003711,1,0.414391,HIGH_PRIORITY_CTR_GAP,"Review first — large gap, meaningful traffic."
5,client_0797ff3a1fc9a6a5,content_7beb639d1052e49e,0.0,311.0,8.540193,10,LOW,informational,0.000000,page_1,...,1.0,0.0,0.0,2.397895,0.003711,0.003711,1,0.449137,HIGH_PRIORITY_CTR_GAP,"Review first — large gap, meaningful traffic."
6,client_0797ff3a1fc9a6a5,content_a0a6b37ae2f9a09c,0.0,248.0,21.725806,0,LOW,informational,0.000000,page_3_5,...,0.0,1.0,0.0,0.000000,0.001369,0.001369,0,0.000151,NOT_EVALUABLE_LABEL_LIMITATION,Manual review only — model score not valid for this tier.
8,client_0797ff3a1fc9a6a5,content_be06033d30b49299,1.0,2092.0,53.148184,0,LOW,informational,0.000478,page_3_5,...,0.0,1.0,0.0,0.000000,0.001369,0.000891,0,0.000664,NOT_EVALUABLE_LABEL_LIMITATION,Manual review only — model score not valid for this tier.
9,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,0.0,240.0,7.520833,30,LOW,informational,0.000000,page_1,...,1.0,0.0,0.0,3.433987,0.003711,0.003711,1,0.454233,HIGH_PRIORITY_CTR_GAP,"Review first — large gap, meaningful traffic."
23586,client_2b4306c3ed003f01,content_3c922fabec32f5df,0.0,239.0,10.422594,0,LOW,informational,0.000000,striking,...,0.0,0.0,1.0,0.000000,0.003290,0.003290,1,0.390783,SEVERE_CTR_GAP_LOW_VOLUME,"Review when time allows — large gap, low traffic."
23587,client_2b4306c3ed003f01,content_97546ac8f303415a,0.0,270.0,8.740741,0,LOW,informational,0.000000,page_1,...,1.0,0.0,0.0,0.000000,0.003711,0.003711,1,0.412499,SEVERE_CTR_GAP_LOW_VOLUME,"Review when time allows — large gap, low traffic."
23588,client_2b4306c3ed003f01,content_bfa8b85d598b0497,0.0,206.0,4.781553,0,LOW,informational,0.000000,page_1,...,1.0,0.0,0.0,0.000000,0.003711,0.003711,1,0.368075,SEVERE_CTR_GAP_LOW_VOLUME,"Review when time allows — large gap, low traffic."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*  

**Intended use.** This queue is meant for a content strategist or SEO reviewer to prioritize
which pages to look at first, not to trigger any automatic changes. It's a starting point for
human review, ranked by likely opportunity — not a verified list of underperforming pages.

**Limits.**  
- A single train/test split's precision@20 ranged from 40% to 95% across five seeds in w06 (mean ~56%), so this month's ranking should be treated as one plausible ordering, not a precise or stable one.  
- Most of the model's apparent skill comes from `position_tier`, which was also used to help
  construct the label itself (`expected_ctr` is looked up per tier). This means the model is
  partly repeating the label's own construction rather than finding fully independent signal.
- `page_3_5` pages can never be labeled high-gap under the current design, regardless of real
  performance — they're excluded from scoring and flagged for manual review instead.
- This queue reflects a single month of data (March 2026). It says nothing about how a page's
  performance is trending, and should not be treated as a forecast.

Given these limits, the queue is best used as a shortlist to investigate, not as a ranked
verdict on which pages are actually underperforming.

In [50]:
display(test_df_E["reason_code"].value_counts())

seeds_data["model_E_precision_at_20"] = []
for seed in seeds_data["seed"]:
    gss = GroupShuffleSplit(test_size =0.2, n_splits = 1, random_state = seed)
    train_idx, test_idx = next(gss.split(X,y, groups= groups))
    X_E_train_per_seed = X.iloc[train_idx]
    X_E_test_per_seed = X.iloc[test_idx]
    y_E_train_per_seed = y.iloc[train_idx]
    y_E_test_per_seed = y.iloc[test_idx]
    model_E_per_seed = LogisticRegression(max_iter=1000, random_state=1)
    model_E_per_seed.fit(X_E_train_per_seed, y_E_train_per_seed)
    y_E_pred_proba = model_E_per_seed.predict_proba(X_E_test_per_seed)[:,1]

    test_df_E_per_seed = df.loc[X_E_test_per_seed.index].copy()
    test_df_E_per_seed["is_high_gap"] = y_E_test_per_seed.values
    test_df_E_per_seed["predicted_high_gap"] = y_E_pred_proba
    test_df_model_E_per_seed = test_df_E_per_seed.sort_values("predicted_high_gap", ascending=False)
    top_20_E_per_seed = test_df_model_E_per_seed.head(20)
    seeds_data["model_E_precision_at_20"].append(top_20_E_per_seed["is_high_gap"].sum() / 20)

display(pd.DataFrame(seeds_data).style.hide(axis="index").format("{:.1%}", subset=["model_E_precision_at_20"]).set_properties(**{"text-align": "center"}))

reason_code
NO_ACTION_NEEDED                  7085
HIGH_PRIORITY_CTR_GAP             2851
HIGH_VOLUME_LOW_CTR_GAP           2395
NOT_EVALUABLE_LABEL_LIMITATION    1454
SEVERE_CTR_GAP_LOW_VOLUME         1422
LOW_PRIORITY_CTR_GAP              1255
Name: count, dtype: int64

seed,train_split,test_split,model_E_precision_at_20
2,80.903438%,19.096562%,40.0%
27,80.347122%,19.652878%,55.0%
30,80.777389%,19.222611%,60.0%
38,79.609003%,20.390997%,40.0%
49,80.047754%,19.952246%,95.0%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*  

**Before acting on any row, a reviewer should:**
- Open the actual page and confirm the low-CTR pattern still holds — the data is a March 2026
  snapshot, so a page could already be fixed, redesigned, or affected by something recent the
  model can't see.
- Check whether `search_volume` and `competition_level` look current — these come from a
  keyword tool, not live search data, and can go stale.
- Treat `HIGH_PRIORITY_CTR_GAP` pages as a shortlist to investigate first, not a confirmed
  diagnosis — remember the ranking itself swung 40-95% precision@20 across seeds in w06.
- For `page_3_5` pages, don't skip review just because the model shows `is_high_gap == 0` —
  that value means "not evaluable," not "confirmed fine."

**Never automate:**
- Auto-publishing or auto-editing any page's title, snippet, or content based on this model's
  output alone.
- Treating a `page_3_5` page's non-flag as evidence the page is performing fine.
- Using a single split's ranking as a final priority order without human review — precision at
  the top of the queue is not stable enough across seeds to justify skipping that step.

In [51]:
display(test_df_E["reason_code"].value_counts())

columns = ["content_hash_id","total_clicks", "total_impressions", "weighted_avg_position", "position_tier", "search_volume", "competition_level", "ctr", "ctr_gap", "is_high_gap"]
display(test_df_E[columns].query("position_tier == 'page_3_5'").sort_values("ctr_gap", ascending=False).head(10).style.set_properties(**{"text-align": "center"}))

reason_code
NO_ACTION_NEEDED                  7085
HIGH_PRIORITY_CTR_GAP             2851
HIGH_VOLUME_LOW_CTR_GAP           2395
NOT_EVALUABLE_LABEL_LIMITATION    1454
SEVERE_CTR_GAP_LOW_VOLUME         1422
LOW_PRIORITY_CTR_GAP              1255
Name: count, dtype: int64

,content_hash_id,total_clicks,total_impressions,weighted_avg_position,position_tier,search_volume,competition_level,ctr,ctr_gap,is_high_gap
27229,content_00278913e26ee5e9,0.000000,702.000000,46.544160,page_3_5,10,HIGH,0.000000,0.001369,0
61526,content_fed5d8e02f5079e6,0.000000,376.000000,50.279255,page_3_5,10,LOW,0.000000,0.001369,0
6,content_a0a6b37ae2f9a09c,0.000000,248.000000,21.725806,page_3_5,0,LOW,0.000000,0.001369,0
27234,content_004e9a270e85c4da,0.000000,221.000000,75.140271,page_3_5,0,LOW,0.000000,0.001369,0
26299,content_0e79f2c556f668b7,0.000000,398.000000,22.304020,page_3_5,260,LOW,0.000000,0.001369,0
26338,content_199ccff5416079a4,0.000000,212.000000,38.872642,page_3_5,90,LOW,0.000000,0.001369,0
26618,content_5d11c5875e8379e8,0.000000,806.000000,26.621588,page_3_5,10,LOW,0.000000,0.001369,0
26737,content_7a7adff45cf53d20,0.000000,358.000000,22.142458,page_3_5,0,LOW,0.000000,0.001369,0
26739,content_7a9efea0583c20ea,0.000000,298.000000,22.744966,page_3_5,10,LOW,0.000000,0.001369,0
26789,content_88c4ecb4dfecbea9,0.000000,331.000000,20.416918,page_3_5,0,LOW,0.000000,0.001369,0


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*  

- **Data age:** this queue is built from one month (March 2026). It should be treated as stale
  and re-run once ~60-90 days have passed, or sooner if the business has reason to expect
  ranking volatility.
- **Benchmark drift:** if a re-run's `expected_ctr_per_tier` moves meaningfully from this run's
  values (page_1: 0.003376, striking: 0.003182, page_3_5: 0.001364), that signals the underlying
  CTR baselines have shifted and the model should be refit, not just re-scored.
- **Growing blind spot:** if the share of pages in `page_3_5` grows significantly month over
  month, more content is falling into the tier this model can't evaluate at all — that's a
  signal to prioritize redesigning the label threshold, not just re-running the current one.
- **Ranking instability:** w06 showed precision@20 ranging 40-95% across seeds on one month of
  data. If a fresh month's re-run also lands consistently near the low end across multiple
  seeds (not just one unlucky split), that's a sign the model's real-world usefulness has
  dropped, not just sampling noise.

In [52]:
con.sql("SELECT MIN(report_date) AS earliest_date, MAX(report_date) AS latest_date from fact_content_daily_performance_table").show()
display(expected_ctr_per_tier)

#remember: NOT_EVALUABLE_LABEL_LIMITATION is for page_3_5 position tier
display(test_df_E["reason_code"].value_counts())
display(pd.DataFrame(seeds_data).style.hide(axis="index").format("{:.1%}", subset=["model_E_precision_at_20"]).set_properties(**{"text-align": "center"}))

┌───────────────┬─────────────┐
│ earliest_date │ latest_date │
│     date      │    date     │
├───────────────┼─────────────┤
│ 2026-03-01    │ 2026-03-31  │
└───────────────┴─────────────┘



position_tier
page_1      0.003711
striking    0.003290
page_3_5    0.001369
dtype: float64

reason_code
NO_ACTION_NEEDED                  7085
HIGH_PRIORITY_CTR_GAP             2851
HIGH_VOLUME_LOW_CTR_GAP           2395
NOT_EVALUABLE_LABEL_LIMITATION    1454
SEVERE_CTR_GAP_LOW_VOLUME         1422
LOW_PRIORITY_CTR_GAP              1255
Name: count, dtype: int64

seed,train_split,test_split,model_E_precision_at_20
2,80.903438%,19.096562%,40.0%
27,80.347122%,19.652878%,55.0%
30,80.777389%,19.222611%,60.0%
38,79.609003%,20.390997%,40.0%
49,80.047754%,19.952246%,95.0%


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [53]:
import json
filtered_test_df_model_E.to_csv("../outputs/action_queue.csv", index=False)

metrics = {
    "expected_ctr_per_tier": expected_ctr_per_tier.to_dict(),
    "reason_code_counts": test_df_E["reason_code"].value_counts().to_dict(),
    "w06_precision_at_20_range": pd.Series(seeds_data["model_E_precision_at_20"]).describe().loc[["min", "max", "mean"]].to_dict(),
    "data_month": "2026-03",
}

# os.makedirs("work/figures", exist_ok=True)  # if you end up wanting this folder too
with open("../outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.